# RuleShift Benchmark v1 — Kaggle Benchmark

## Runtime bootstrap

In [ ]:
import sys
from pathlib import Path

import kaggle_benchmarks as kbench


def _find_repo_root() -> Path:
    candidates: list[Path] = []
    seen: set[Path] = set()

    for origin in (Path.cwd().resolve(),):
        for candidate in (origin, *origin.parents):
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for search_root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not search_root.exists():
            continue
        for manifest_path in search_root.rglob("frozen_artifacts_manifest.json"):
            candidate = manifest_path.parents[2]
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for candidate in candidates:
        if (candidate / "src").is_dir() and (
            candidate / "packaging" / "kaggle" / "frozen_artifacts_manifest.json"
        ).is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate repo root. Expected src/ and "
        "packaging/kaggle/frozen_artifacts_manifest.json."
    )


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))


## Frozen leaderboard split

In [ ]:
from core.kaggle import (
    build_kaggle_payload,
    load_leaderboard_dataframe,
    normalize_count_result_df,
    run_binary_task,
    validate_kaggle_payload,
)

PRIVATE_DATASET_ROOT, frozen_splits, leaderboard_df = load_leaderboard_dataframe(
    repo_root=REPO_ROOT,
)


## Official task

In [ ]:
@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
)
def ruleshift_benchmark_v1_binary(
    llm,
    prompt_binary: str,
    probe_targets: tuple,
    split: str,
    episode_id: str | None = None,
) -> tuple[int, int]:
    del split
    del episode_id
    return run_binary_task(
        llm=llm,
        prompt_binary=prompt_binary,
        probe_targets=probe_targets,
    )


## Official payload

In [ ]:
import json

binary_results = ruleshift_benchmark_v1_binary.evaluate(
    llm=[kbench.llm],
    evaluation_data=leaderboard_df[["episode_id", "split", "prompt_binary", "probe_targets"]],
)
binary_df = normalize_count_result_df(binary_results.as_dataframe(), dataframe_label="binary_df")
payload = build_kaggle_payload(binary_df=binary_df)
validate_kaggle_payload(payload)

print("=== RuleShift Benchmark v1 — Canonical Result ===")
print(json.dumps(payload, indent=2))
print("=== End Canonical Result ===")


## Task selection

In [ ]:
%choose ruleshift_benchmark_v1_binary